# 🏗️ ArchAI — Blender + BlenderLLM Server

**AI-архитектор: текст → 3D-модель в Blender**

Интеграция:
- [BlenderLLM](https://github.com/FreedomIntelligence/BlenderLLM) — LLM генерирует bpy скрипты из текста
- [BlenderProc](https://github.com/DLR-RM/BlenderProc) — фотореалистичный рендеринг
- [StableGen](https://github.com/sakalond/StableGen) — AI-текстурирование

---

**Пайплайн:** Пользователь пишет текст → BlenderLLM генерирует bpy скрипт → Blender выполняет → 3D модель/рендер

## 📦 Шаг 1: Установка

In [ ]:
#@title Установка Blender + зависимости { display-mode: "form" }
!apt-get update -qq && apt-get install -y -qq blender > /dev/null 2>&1
!pip install -q flask flask-cors pyngrok httpx transformers torch accelerate sentencepiece

# Проверка
!blender --version
import torch
print(f"\n✅ Blender установлен | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
#@title Загрузка скриптов ArchAI { display-mode: "form" }
!mkdir -p /content/archai/output

REPO = "https://raw.githubusercontent.com/smartmoneymoscow-cell/AI_Arhitector/main/blender"
!curl -sL "{REPO}/blenderllm_bridge.py" -o /content/archai/blenderllm_bridge.py
!curl -sL "{REPO}/generate_building.py" -o /content/archai/generate_building.py
!curl -sL "{REPO}/render_interior.py" -o /content/archai/render_interior.py
!curl -sL "{REPO}/server.py" -o /content/archai/server.py

print("✅ Скрипты загружены")
!ls -la /content/archai/*.py

## 🤖 Шаг 2: Загрузка BlenderLLM модели

In [ ]:
#@title Загрузка модели BlenderLLM (Qwen2.5-Coder-7B) { display-mode: "form" }

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "FreedomIntelligence/BlenderLLM"

print(f"⏳ Загрузка модели {MODEL_ID}...")
print("   (занимает ~3-5 минут, модель ~14GB)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\n✅ Модель загружена!")
print(f"   Device: {model.device}")
print(f"   VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 🧪 Шаг 3: Тест генерации bpy скрипта

In [ ]:
#@title Генерация bpy скрипта из текста { display-mode: "form" }

SYSTEM_PROMPT = """You are an expert in using bpy script to create 3D models. Based on the following instruction, your task is to write the corresponding bpy script that will generate the desired 3D model in Blender. Please pay close attention to every detail in the script and ensure it fully adheres to the provided specifications."""

user_prompt = "Двухэтажный кирпичный дом 10x12 с двускатной кровлей"  #@param {type:"string"}
max_tokens = 2048  #@param {type:"integer"}

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(f"⏳ Генерация скрипта для: {user_prompt}")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated = output[0][len(inputs.input_ids[0]):]
script = tokenizer.decode(generated, skip_special_tokens=True)

# Clean markdown artifacts
import re
script = re.sub(r'```python\s*', '', script)
script = re.sub(r'```\s*', '', script)
script = script.strip()

print(f"\n✅ Скрипт сгенерирован ({len(script)} символов)")
print(f"\n{'='*60}")
print(script[:2000])
print(f"{'='*60}")

## 🏠 Шаг 4: Выполнение в Blender

In [ ]:
#@title Запуск bpy скрипта в Blender { display-mode: "form" }

export_format = "glb"  #@param ["glb", "fbx", "obj", "blend"]
render_preview = True  #@param {type:"boolean"}

import os

# Save script
script_path = "/content/archai/output/generated_script.py"
with open(script_path, 'w') as f:
    f.write("import bpy\nimport os\nimport math\n\n")
    f.write("bpy.ops.object.select_all(action='SELECT')\nbpy.ops.object.delete()\n\n")
    f.write(script)

output_file = f"/content/archai/output/building.{export_format}"

# Add export command
with open(script_path, 'a') as f:
    if export_format in ('glb', 'gltf'):
        f.write(f"\nbpy.ops.export_scene.gltf(filepath=r'{output_file}', export_format='GLB')\n")
    elif export_format == 'fbx':
        f.write(f"\nbpy.ops.export_scene.fbx(filepath=r'{output_file}')\n")
    elif export_format == 'obj':
        f.write(f"\nbpy.ops.wm.obj_export(filepath=r'{output_file}')\n")
    elif export_format == 'blend':
        f.write(f"\nbpy.ops.wm.save_as_mainfile(filepath=r'{output_file}')\n")
    
    if render_preview:
        preview = output_file.rsplit('.', 1)[0] + '_preview.png'
        f.write(f"""
# Render preview
scene = bpy.context.scene
scene.render.engine = 'CYCLES'
scene.cycles.samples = 64
scene.render.resolution_x = 1280
scene.render.resolution_y = 720
scene.render.image_settings.file_format = 'PNG'
scene.render.filepath = r'{preview}'
bpy.ops.render.render(write_still=True)
""")

print(f"⏳ Выполнение в Blender...")
!blender --background --factory-startup --python {script_path} 2>&1 | tail -20

# Show results
if os.path.exists(output_file):
    print(f"\n✅ Модель: {output_file} ({os.path.getsize(output_file)/1024:.1f} KB)")
else:
    print("\n❌ Ошибка экспорта")

if render_preview:
    preview = output_file.rsplit('.', 1)[0] + '_preview.png'
    if os.path.exists(preview):
        from IPython.display import Image, display
        print(f"📸 Превью:")
        display(Image(preview, width=800))
    else:
        print("⚠️ Превью не отрендерено")

## 🛋️ Шаг 5: Фотореалистичный интерьер

In [ ]:
#@title Рендер интерьера { display-mode: "form" }

room_type = "living_room"  #@param ["living_room", "bedroom", "kitchen", "office"]
room_width = 6  #@param {type:"slider", min:3, max:15, step:1}
room_length = 8  #@param {type:"slider", min:3, max:15, step:1}
room_height = 3  #@param {type:"slider", min:2.5, max:5, step:0.5}
interior_style = "modern"  #@param ["modern", "classic", "scandinavian", "loft", "minimalist"]

import json

furniture_map = {
    "living_room": ["sofa", "table", "chandelier"],
    "bedroom": ["bed", "chandelier"],
    "kitchen": ["table", "chandelier"],
    "office": ["table", "chandelier"],
}

params = {
    "room_type": room_type,
    "width": room_width,
    "length": room_length,
    "height": room_height,
    "style": interior_style,
    "furniture": furniture_map.get(room_type, []),
    "camera_position": "corner",
    "samples": 256,
    "resolution": 1920,
}

with open('/content/archai/interior_params.json', 'w') as f:
    json.dump(params, f, indent=2)

print(f"⏳ Рендер интерьера: {room_type}, {interior_style}, {room_width}x{room_length}m")
!blender --background --factory-startup --python /content/archai/render_interior.py -- /content/archai/interior_params.json /content/archai/output/interior.png 2>&1 | tail -5

if os.path.exists('/content/archai/output/interior.png'):
    from IPython.display import Image, display
    print("\n✅ Интерьер отрендерен!")
    display(Image('/content/archai/output/interior.png', width=900))
else:
    print("❌ Ошибка рендера")

## 🌐 Шаг 6: API сервер для веб-интерфейса

In [ ]:
#@title Запуск ArchAI Blender Server { display-mode: "form" }

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
import subprocess, uuid, sys, os

sys.path.insert(0, '/content/archai')
from blenderllm_bridge import BlenderLLMBridge

app = Flask(__name__)
CORS(app)

OUTPUT = "/content/archai/output"
os.makedirs(OUTPUT, exist_ok=True)

# Инициализация моста с загруженной моделью
bridge = BlenderLLMBridge(
    model_path="FreedomIntelligence/BlenderLLM",
    blender_path="blender",
    anthropic_key=os.environ.get("ANTHROPIC_API_KEY", ""),
)
bridge._model = model
bridge._tokenizer = tokenizer
bridge._model_loaded = True

@app.route('/health')
@app.route('/api/v1/health')
def health():
    return jsonify({"status":"ok","service":"archai-blender","model":True})

@app.route('/api/v1/generate/building', methods=['POST'])
def gen_building():
    data = request.json or {}
    prompt = data.get('prompt','')
    fmt = data.get('export_format','glb')
    if not prompt: return jsonify({"error":"prompt required"}), 400
    
    job = uuid.uuid4().hex[:8]
    outf = f"{OUTPUT}/{job}.{fmt}"
    
    try:
        script = bridge.generate(prompt)
        result = bridge.run_in_blender(script, outf, export_format=fmt)
        if os.path.exists(outf):
            return send_file(outf, as_attachment=True,
                            download_name=f"archai_{job}.{fmt}")
        return jsonify({"error":"failed","details":result}), 500
    except Exception as e:
        return jsonify({"error":str(e)}), 500

@app.route('/api/v1/generate/script', methods=['POST'])
def gen_script():
    data = request.json or {}
    prompt = data.get('prompt','')
    if not prompt: return jsonify({"error":"prompt required"}), 400
    try:
        script = bridge.generate(prompt)
        return jsonify({"script":script,"chars":len(script)})
    except Exception as e:
        return jsonify({"error":str(e)}), 500

@app.route('/api/v1/render/interior', methods=['POST'])
def render_int():
    data = request.json or {}
    job = uuid.uuid4().hex[:8]
    pf = f"{OUTPUT}/{job}_int.json"
    of = f"{OUTPUT}/{job}_int.png"
    with open(pf,'w') as f: json.dump(data,f)
    r = subprocess.run(['blender','--background','--factory-startup',
        '--python','/content/archai/render_interior.py','--',pf,of],
        capture_output=True, text=True, timeout=300)
    if os.path.exists(of):
        return send_file(of, as_attachment=True)
    return jsonify({"error":r.stderr[-300:]}), 500

# Запуск
public_url = ngrok.connect(5000).public_url
print(f"\n{'='*60}")
print(f"🌐 ArchAI Blender Server запущен!")
print(f"📡 URL: {public_url}")
print(f"🤖 BlenderLLM: загружена")
print(f"{'='*60}")
print(f"\nEndpoints:")
print(f"  GET  {public_url}/health")
print(f"  POST {public_url}/api/v1/generate/building")
print(f"  POST {public_url}/api/v1/generate/script")
print(f"  POST {public_url}/api/v1/render/interior")
print(f"\n💡 Скопируйте URL в веб-интерфейс ArchAI")

app.run(port=5000, host='0.0.0.0')

## 📝 Текстовые промты

### Здания:
- `двухэтажный кирпичный дом 10×12 с двускатной кровлей и балконом`
- `современный стеклянный офис 5 этажей 15×20`
- `деревянный коттедж с террасой и гаражом`
- `минималистичная вила с плоской кровлей 12×15`

### Интерьеры:
- `гостиная в скандинавском стиле 6×8`
- `спальня в стиле лофт 4×5`
- `минималистичная кухня 3×4`
- `кабинет с классической мебелью 5×6`